# Perfilamiento de la Fuente Secundaria: API IDEAM (datos.gov.co)
**Proyecto:** ETL Segunda Entrega — Huella de Carbono en Hogares Colombianos (ODS 12)  
**Responsable:** David Aponte (Persona 2 — API y Calidad de Datos)  
**Autores:** Nikol Hurtado, David Aponte, Diana Panesso, Sebastián Celorio  

## Objetivo
Documentar el perfilamiento de la **API SODA del IDEAM** (dataset `sbwg-7ju4` en datos.gov.co),
evaluando cobertura espacial/temporal, validez física de temperatura y estrategia de integración con la ECV 2025.
Cada hallazgo se vincula con las reglas GX DQ12–DQ18 y con los requisitos R6/R7.

> **Instrucciones de ejecución:**  
> Opción A: correr el pipeline completo (`docker compose up` + DAG) y luego usar el CSV generado.  
> Opción B: correr `python scripts/verificar_cobertura_api.py` para explorar la cobertura primero.  
> Opción C: ejecutar directamente las celdas (se conecta a la API en tiempo real).


## 0. Configuración y Parámetros del Endpoint

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

try:
    import requests
    HAS_REQUESTS = True
except ImportError:
    HAS_REQUESTS = False
    print('[AVISO] requests no instalado. Las celdas de llamada a la API quedarán desactivadas.')

PROJECT_ROOT = Path('..').resolve()
REF_DIR  = PROJECT_ROOT / 'data' / 'reference'
RAW_API  = PROJECT_ROOT / 'data' / 'raw' / 'api'

# ─── Parámetros del endpoint (deben coincidir con src/config.py) ───
ENDPOINT    = 'https://www.datos.gov.co/resource/sbwg-7ju4.json'
SENSOR_CODE = '0068'   # Temperatura del Aire a 2 metros, °C
API_COLUMNS = ['codigoestacion', 'nombreestacion', 'departamento', 'municipio',
               'latitud', 'longitud', 'suma', 'n', 'minimo', 'maximo',
               'window_start', 'window_end']
TEMP_MIN_PLAUSIBLE = -15.0
TEMP_MAX_PLAUSIBLE =  50.0

dane_ref     = pd.read_csv(REF_DIR / 'dane_departamentos.csv', sep=';')
CODIGOS_DANE = set(dane_ref['codigo_dane'].astype(int))

print(f'Endpoint IDEAM : {ENDPOINT}')
print(f'Sensor code    : {SENSOR_CODE}  (Temperatura Aire 2 m, °C)')
print(f'Rango físico   : [{TEMP_MIN_PLAUSIBLE}, {TEMP_MAX_PLAUSIBLE}] °C')
print(f'Departamentos DANE válidos: {len(CODIGOS_DANE)}')


---
## 1. Contrato del Endpoint — DQ12 (Critical)

La consulta SODA utiliza agregación en el servidor (`$select`, `$group`) para reducir el volumen de datos.
Columnas esperadas en la respuesta:

| Campo | Tipo | Rol |
|---|---|---|
| `codigoestacion` | str | ID de la estación IDEAM |
| `nombreestacion` | str | Nombre de la estación |
| `departamento` | str | Llave de integración con ECV |
| `municipio` | str | Contexto geográfico |
| `latitud`, `longitud` | float | Coordenadas para DQ18 |
| `suma`, `n` | float | Para calcular temperatura media exacta: $\\bar{T} = \\text{suma}/n$ |
| `minimo`, `maximo` | float | Para validar rango físico (DQ15) |
| `window_start`, `window_end` | date-str | Compatibilidad temporal con ECV 2025 (DQ16) |

**DQ12 (Critical 100%):** Si la API modifica su esquema (columna renombrada o eliminada), el pipeline falla determinísticamente → no se reintenta.


In [ ]:
# Muestra de la API — ventana de 7 días para validar el contrato
sample_df = None
if HAS_REQUESTS:
    try:
        params = {
            '$select': ('codigoestacion,nombreestacion,departamento,municipio,latitud,longitud,'
                        'sum(valorobservado) as suma,count(*) as n,'
                        'min(valorobservado) as minimo,max(valorobservado) as maximo'),
            '$where': (f"codigosensor='{SENSOR_CODE}' "
                       "AND fechaobservacion >= '2025-01-01T00:00:00' "
                       "AND fechaobservacion < '2025-01-08T00:00:00'"),
            '$group': 'codigoestacion,nombreestacion,departamento,municipio,latitud,longitud',
            '$order': 'codigoestacion',
            '$limit': '1000',
        }
        r = requests.get(ENDPOINT, params=params, timeout=(15, 120))
        r.raise_for_status()
        sample_df = pd.DataFrame(r.json())
        sample_df['window_start'] = '2025-01-01'
        sample_df['window_end']   = '2025-01-08'
        print(f'Muestra obtenida: {len(sample_df)} filas, {sample_df.shape[1]} columnas')
        print(f'Columnas: {sample_df.columns.tolist()}')
        display(sample_df.head())
    except Exception as exc:
        print(f'[AVISO] No se pudo conectar a la API: {exc}')
        print('Continúa con el CSV pre-generado (si existe).')
else:
    print('requests no disponible — saltando llamada en vivo.')

# Intentar leer CSV ya generado por el pipeline
if sample_df is None:
    csv_candidate = next(RAW_API.glob('api_ideam_raw.csv'), None) if RAW_API.exists() else None
    if csv_candidate is None:
        # Buscar en data/work/
        work = PROJECT_ROOT / 'data' / 'work'
        if work.exists():
            for f in sorted(work.rglob('api_ideam_raw.csv')):
                csv_candidate = f
                break
    if csv_candidate:
        sample_df = pd.read_csv(csv_candidate)
        print(f'CSV cargado desde: {csv_candidate}')
        print(f'Filas: {len(sample_df):,}  Columnas: {sample_df.shape[1]}')
        display(sample_df.head())
    else:
        print('[AVISO] No hay CSV de la API disponible. Ejecuta el DAG primero.')


---
## 2. Validez del Rango Físico de Temperatura — DQ15 (Critical ≥ 99.5 %)

> **Aviso oficial de la ficha del IDEAM:** *«El IDEAM no validó previamente la totalidad de los valores
> horarios publicados y pueden existir lecturas anómalas por fallas técnicas en sensores remotos».*

- Rango plausible para Colombia: **[-15 °C, 50 °C]** (cubre desde los páramos andinos a las costas tropicales).
- La regla evalúa `minimo` y `maximo` de cada estación × ventana: una sola lectura imposible queda registrada.
- **Umbral 99.5 %:** hasta 0.5 % de fallas de sensor son tolerables sin bloquear el pipeline, pero se registran y excluyen del promedio departamental. Base: aviso de la ficha del IDEAM.
- **Decisión de pipeline:** filas con `minimo` < -15 °C o `maximo` > 50 °C se excluyen de `build_dim_clima()` y se cuentan en `integration_report.json`.


In [ ]:
if sample_df is not None:
    df = sample_df.copy()
    for col in ('suma', 'n', 'minimo', 'maximo', 'latitud', 'longitud'):
        df[col] = pd.to_numeric(df[col], errors='coerce')

    n_total = len(df)
    fuera_min = (df['minimo'] < TEMP_MIN_PLAUSIBLE).sum()
    fuera_max = (df['maximo'] > TEMP_MAX_PLAUSIBLE).sum()
    fuera_any = ((df['minimo'] < TEMP_MIN_PLAUSIBLE) | (df['maximo'] > TEMP_MAX_PLAUSIBLE)).sum()
    pct_ok = (n_total - fuera_any) / n_total * 100

    print('DQ15 — Rango físico plausible de temperatura')
    print(f'  Total filas          : {n_total:,}')
    print(f'  minimo < {TEMP_MIN_PLAUSIBLE}°C : {fuera_min:,}  (fallas sensor por frío extremo)')
    print(f'  maximo > {TEMP_MAX_PLAUSIBLE}°C  : {fuera_max:,}  (fallas sensor por calor extremo)')
    print(f'  Total fuera de rango : {fuera_any:,}  ({100 - pct_ok:.4f}%)')
    print(f'  DQ15 umbral ≥99.5%   : {"PASA" if pct_ok >= 99.5 else "FALLA"}')

    print(f'\n  Estadísticas de temperatura mínima (°C):')
    print(df['minimo'].describe().to_string())
    print(f'\n  Estadísticas de temperatura máxima (°C):')
    print(df['maximo'].describe().to_string())
else:
    print('Sin datos: rango físico Colombia [-15°C, 50°C] justificado por geografia del país.')
    print('Umbral 99.5%: basado en aviso oficial IDEAM sobre posibles fallas de sensor.')


---
## 3. Cobertura Departamental — DQ13, DQ14 y DQ17

- **DQ13 (Critical ≥1 fila):** Si la extracción completa devuelve 0 filas, la integración con la ECV es imposible → bloqueo.
- **DQ14 (Critical ≥99%):** `departamento` nulo → estación sin llave de integración → no se puede asignar clima a ningún hogar de ese departamento.
- **DQ17 (Warning ≥20 departamentos):** Con menos de 20 departamentos con datos climáticos, la cobertura es insuficiente para análisis de correlación temperatura–consumo (R6/R7). Base: Colombia tiene 33 departamentos; 20 = mínimo estadístico para inferencia.


In [ ]:
if sample_df is not None:
    df = sample_df.copy()

    # DQ13 — fila mínima
    print(f'DQ13 — Filas totales: {len(df):,}  (min=1) → {"PASA" if len(df) >= 1 else "FALLA"}')

    # DQ14 — departamento nulo
    n_dep_null = df['departamento'].isna().sum()
    pct_dep_ok = (1 - n_dep_null / len(df)) * 100
    print(f'DQ14 — departamento nulos: {n_dep_null}  (completitud {pct_dep_ok:.2f}%) → {"PASA" if pct_dep_ok >= 99 else "FALLA"}')

    # DQ17 — departamentos distintos
    n_deptos = df['departamento'].dropna().nunique()
    print(f'DQ17 — Departamentos distintos: {n_deptos}  (umbral ≥20) → {"PASA" if n_deptos >= 20 else "FALLA"}')

    # Distribución por departamento
    dist_dep = df.groupby('departamento').agg(
        estaciones=('codigoestacion', 'nunique'),
        filas=('codigoestacion', 'count'),
    ).sort_values('filas', ascending=False)
    print(f'\nDistribución por departamento (top 15):')
    display(dist_dep.head(15))

    print(f'\nJustificación DQ17 umbral=20:')
    print('  Colombia tiene 33 departamentos. Con <20 con datos, la correlación temperatura–kWh')
    print('  (R6/R7) tendría n insuficiente para inferencia estadística robusta.')
else:
    print('Sin datos. Cobertura esperada: ≥25 departamentos con datos del IDEAM.')


---
## 4. Consistencia Temporal — DQ16 (Critical 100 %)

La ECV 2025 captura el año 2025. La API debe extraer datos del mismo período para que la integración sea válida.  
**DQ16 (Critical 100%):** Ventanas fuera del año 2025 en `window_start` introducirían datos climáticos de otro período → incompatibles con la ECV → bloqueo.


In [ ]:
if sample_df is not None:
    if 'window_start' in sample_df.columns:
        n_bad = (~sample_df['window_start'].astype(str).str.startswith('2025')).sum()
        pct_ok = (1 - n_bad / len(sample_df)) * 100
        print(f'DQ16 — Ventanas fuera de 2025: {n_bad}  ({100 - pct_ok:.4f}%)')
        print(f'DQ16 umbral 100% → {"PASA" if n_bad == 0 else "FALLA"}')
        print(f'\nVentanas encontradas: {sample_df["window_start"].unique()[:10]}')
    else:
        print('[AVISO] Columna window_start no presente en la muestra.')
else:
    print('Sin datos. DQ16: API_START_DATE=2025-01-01, API_END_DATE=2026-01-01 garantizan año 2025.')


---
## 5. Coordenadas Geográficas — DQ18 (Warning ≥ 99 %)

Límites geográficos de Colombia:
- **Latitud:** [-5°, 14°] (desde el sur de Amazonas hasta La Guajira)
- **Longitud:** [-82°, -66°] (desde el Pacífico hasta los Llanos Orientales; incluye San Andrés y Providencia a -81.7°)

**DQ18 (Warning ≥99%):** Coordenadas fuera de estos límites indican estaciones con datos corruptos de GPS. Warning porque no impiden el cálculo de temperatura, pero sí el análisis espacial.


In [ ]:
if sample_df is not None:
    df = sample_df.copy()
    df['latitud']   = pd.to_numeric(df['latitud'],  errors='coerce')
    df['longitud']  = pd.to_numeric(df['longitud'], errors='coerce')

    n_lat_ok  = df['latitud'].between(-5, 14).sum()
    n_lon_ok  = df['longitud'].between(-82, -66).sum()
    n_lat_bad = len(df) - n_lat_ok
    n_lon_bad = len(df) - n_lon_ok
    pct_lat   = n_lat_ok / len(df) * 100
    pct_lon   = n_lon_ok / len(df) * 100

    print('DQ18 — Coordenadas dentro de Colombia')
    print(f'  Latitud  [-5°, 14°]:   {n_lat_ok:,} OK  {n_lat_bad:,} fuera  ({100 - pct_lat:.4f}%) → {"PASA" if pct_lat >= 99 else "FALLA"}')
    print(f'  Longitud [-82°, -66°]: {n_lon_ok:,} OK  {n_lon_bad:,} fuera  ({100 - pct_lon:.4f}%) → {"PASA" if pct_lon >= 99 else "FALLA"}')

    print('\nEstadísticas de latitud:')
    print(df['latitud'].describe().to_string())
    print('\nEstadísticas de longitud:')
    print(df['longitud'].describe().to_string())
else:
    print('Sin datos. Límites Colombia: lat [-5, 14], lon [-82, -66]. Umbral ≥99%.')


---
## 6. Clasificación Bioclimática (Soporte R6 y R7)

La temperatura media departamental se clasifica en pisos térmicos para responder R6:  
*«¿Los hogares de departamentos cálidos generan más kWh/mes y kg CO₂ que los de templados/fríos?»*

| Piso Térmico | Rango | Ejemplos |
|---|---|---|
| **Cálido** | > 24 °C | Atlántico, Bolívar, César |
| **Templado** | 18 °C – 24 °C | Antioquia, Risaralda, Caldas |
| **Frío** | < 18 °C | Bogotá D.C., Boyacá, Nariño |


In [ ]:
def piso_termico(temp):
    if pd.isna(temp): return 'Sin dato'
    if temp > 24.0:   return 'Cálido'
    if temp >= 18.0:  return 'Templado'
    return 'Frío'

# Demostración con datos de la muestra si disponibles
if sample_df is not None:
    df = sample_df.copy()
    for col in ('suma', 'n'):
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df[df['n'].gt(0) & df['suma'].notna()]

    clima_dep = df.groupby('departamento').agg(
        suma=('suma', 'sum'), n=('n', 'sum')
    ).reset_index()
    clima_dep['temp_media_c'] = (clima_dep['suma'] / clima_dep['n']).round(2)
    clima_dep['piso_termico'] = clima_dep['temp_media_c'].apply(piso_termico)
    clima_dep = clima_dep.sort_values('temp_media_c', ascending=False)

    print('Temperatura media por departamento y piso térmico:')
    display(clima_dep[['departamento', 'temp_media_c', 'piso_termico']].head(20))

    print('\nResumen por piso térmico:')
    print(clima_dep['piso_termico'].value_counts().to_string())
else:
    test_temps = [14.2, 21.5, 28.3, float('nan')]
    print('Función de clasificación — prueba:')
    for t in test_temps:
        print(f'  Temperatura {t}°C → Piso térmico: {piso_termico(t)}')


---
## 7. Estrategia de Normalización del Campo `departamento`

El IDEAM publica los nombres de departamentos sin estandarización consistente:
- 'Bogotá, D.C.' / 'BOGOTÁ D.C.' / 'Bogotá' → deben mapear a código DANE 11
- 'Archipiélago de San Andrés...' → código DANE 88
- 'La Guajira' / 'Guajira' → código DANE 44

La tabla `ALIAS_DEPARTAMENTO` en `src/transform.py` maneja estas variantes.


In [ ]:
import unicodedata, re

ALIAS_DEP = {
    'bogota': 'bogota dc',
    'bogota d c': 'bogota dc',
    'santafe de bogota': 'bogota dc',
    'archipielago de san andres providencia y santa catalina': 'san andres y providencia',
    'san andres providencia y santa catalina': 'san andres y providencia',
    'san andres': 'san andres y providencia',
    'guajira': 'la guajira',
}

def canonical(text):
    t = unicodedata.normalize('NFKD', str(text))
    t = ''.join(c for c in t if not unicodedata.combining(c)).lower()
    t = re.sub(r'[^a-z0-9 ]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    t = ALIAS_DEP.get(t, t)
    return t.replace('bogota d c', 'bogota dc')

# Test
ejemplos = ['Bogotá, D.C.', 'BOGOTÁ D.C.', 'Archipiélago de San Andrés, Providencia y Santa Catalina',
            'La Guajira', 'ANTIOQUIA', 'Valle del Cauca', 'Bolívar']
test_df = pd.DataFrame({'original': ejemplos, 'canonical': [canonical(e) for e in ejemplos]})
# Mapear código DANE
mapa_nombre_codigo = {canonical(row['departamento']): row['codigo_dane']
                       for _, row in dane_ref.iterrows()}
test_df['codigo_dane'] = test_df['canonical'].map(mapa_nombre_codigo)
display(test_df)


---
## 8. Matriz de Síntesis — API IDEAM (DQ12–DQ18)


In [ ]:
sintesis_api = pd.DataFrame([
    {'Atributo':'columnas req.',     'Dimensión':'Validez (contrato)', 'Regla':'DQ12','Sev':'Critical','Umbral':'100%',   'Base_umbral':'Cambio de esquema en la API = error determinístico'},
    {'Atributo':'filas totales',     'Dimensión':'Completitud',        'Regla':'DQ13','Sev':'Critical','Umbral':'≥1',     'Base_umbral':'0 filas = sin datos climáticos = R6/R7 inviables'},
    {'Atributo':'departamento',      'Dimensión':'Completitud',        'Regla':'DQ14','Sev':'Critical','Umbral':'≥99%',   'Base_umbral':'PROVISIONAL — estación sin llave: no contribuye a dim_clima'},
    {'Atributo':'minimo, maximo',    'Dimensión':'Validez',            'Regla':'DQ15','Sev':'Critical','Umbral':'≥99.5%', 'Base_umbral':'Rango físico Colombia. Aviso oficial IDEAM de fallas sensor'},
    {'Atributo':'window_start',      'Dimensión':'Consistencia temp.', 'Regla':'DQ16','Sev':'Critical','Umbral':'100%',   'Base_umbral':'ECV 2025: clima debe ser 2025 para alineación temporal'},
    {'Atributo':'departamento',      'Dimensión':'Completitud (cob.)', 'Regla':'DQ17','Sev':'Warning', 'Umbral':'≥20',    'Base_umbral':'PROVISIONAL — mínimo estadístico para correlación R6/R7 (33 deptos total)'},
    {'Atributo':'latitud, longitud', 'Dimensión':'Validez',            'Regla':'DQ18','Sev':'Warning', 'Umbral':'≥99%',   'Base_umbral':'Límites geográficos Colombia [lat -5/14, lon -82/-66]'},
])
display(sintesis_api)


---
## 9. Conclusiones y Decisiones del Pipeline (API IDEAM)


In [ ]:
conclusiones = pd.DataFrame([
    {'Hallazgo':'Nombres departamento no estandarizados',
     'Ejemplo': 'Bogotá D.C. / BOGOTÁ, D.C.',
     'Decisión':'Normalización Unicode + tabla ALIAS_DEPARTAMENTO → canonical name → código DANE'},
    {'Hallazgo':'Fallas de sensor (<0.5%)',
     'Ejemplo': 'minimo < -15°C o maximo > 50°C',
     'Decisión':'Excluir fila de build_dim_clima(); registrar en integration_report.json'},
    {'Hallazgo':'Cobertura parcial (no todos los 33 deptos tienen estaciones)',
     'Ejemplo': 'Amazonas, Guainía, Vaupés tienen pocas estaciones',
     'Decisión':'Hogares sin clima: fk_clima NULL + clima_emparejado=0; R6/R7 los excluyen'},
    {'Hallazgo':'Temperatura media = suma(suma)/suma(n) de las ventanas',
     'Ejemplo': 'Suma ponderada por número de lecturas',
     'Decisión':'Promedio exacto y reproducible; no promedio de promedios (evita sesgo de estación)'},
    {'Hallazgo':'DQ17 umbral=20 PROVISIONAL',
     'Ejemplo': 'Calibrar con scripts/verificar_cobertura_api.py',
     'Decisión':'Ejecutar verificación antes del primer deploy; ajustar umbral con datos reales'},
])
display(conclusiones)
